# MiniMax-H3 Image-to-Video (FL2VA) on Google Colab

**Requires:** Colab Pro with A100 (40/80GB) or L4 (24GB). Free T4 will OOM.

Supports resolutions (480p/720p/768p/1080p), aspect ratios, optional last frame, and no content filters.
Run cells in order. Cell 4 prints your public Cloudflare Tunnel URL.

In [ ]:
!pip install -q -U git+https://github.com/huggingface/diffusers.git transformers accelerate safetensors sentencepiece protobuf
!pip install -q fastapi uvicorn python-multipart pillow imageio[ffmpeg]

# Download Cloudflare Tunnel binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
import torch
from diffusers import ComponentsManager, ModularPipeline

# Enable automatic CPU offloading so the 33B model fits in Colab VRAM
manager = ComponentsManager()
manager.enable_auto_cpu_offload(device="cuda", memory_reserve_margin="6GB")

print("Loading MiniMax-H3 FL2VA pipeline...")
pipe = ModularPipeline.from_pretrained(
    "MiniMaxAI/MiniMax-H3",
    components_manager=manager,
)
pipe.load_components(workflow="fl2va", dtype=torch.bfloat16)
print("Model loaded successfully!")

In [ ]:
%%writefile server.py
import io
import os
import uuid
import torch
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from diffusers.utils.export_utils import encode_video
from diffusers import ComponentsManager, ModularPipeline

app = FastAPI(title="MiniMax-H3 I2V API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

manager = ComponentsManager()
manager.enable_auto_cpu_offload(device="cuda", memory_reserve_margin="6GB")
pipe = ModularPipeline.from_pretrained("MiniMaxAI/MiniMax-H3", components_manager=manager)
pipe.load_components(workflow="fl2va", dtype=torch.bfloat16)

OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESOLUTIONS = {
    "480p": 480,
    "720p": 720,
    "768p": 768,
    "1080p": 1080,
}
ASPECT_RATIOS = {
    "auto": None,
    "16:9": 16/9,
    "9:16": 9/16,
    "1:1": 1.0,
    "4:3": 4/3,
    "3:4": 3/4,
    "21:9": 21/9,
}

def snap_frames(seconds: float, fps: int = 24) -> int:
    frames = max(124, round(seconds * fps))
    while (frames - 5) % 17 != 0:
        frames += 1
    return min(frames, 362)

def compute_size(resolution: str, aspect_ratio: str, image: Image.Image):
    short = RESOLUTIONS.get(resolution, 768)
    short = max(32, (short // 32) * 32)
    if aspect_ratio == "auto" or aspect_ratio not in ASPECT_RATIOS:
        ar = image.width / image.height
    else:
        ar = ASPECT_RATIOS[aspect_ratio]
    if ar >= 1:
        height = short
        width = int(round(height * ar / 32) * 32)
    else:
        width = short
        height = int(round(width / ar / 32) * 32)
    return height, width

@app.get("/health")
def health():
    return {"status": "ok", "model": "MiniMax-H3-FL2VA"}

@app.get("/info")
def info():
    return {
        "resolutions": list(RESOLUTIONS.keys()),
        "aspect_ratios": list(ASPECT_RATIOS.keys()),
        "duration_min": 5,
        "duration_max": 15,
        "steps_min": 4,
        "steps_max": 50,
        "supports_last_frame": True,
        "fps": 24,
        "audio": "native stereo",
    }

@app.post("/generate")
async def generate_video(
    image: UploadFile = File(...),
    last_image: UploadFile = File(None),
    prompt: str = Form(...),
    resolution: str = Form("768p"),
    aspect_ratio: str = Form("auto"),
    duration: float = Form(5.0),
    num_inference_steps: int = Form(20),
    seed: int = Form(-1),
):
    try:
        img_bytes = await image.read()
        init_image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        last = None
        if last_image is not None:
            last_bytes = await last_image.read()
            last = Image.open(io.BytesIO(last_bytes)).convert("RGB")

        height, width = compute_size(resolution, aspect_ratio, init_image)
        num_frames = snap_frames(duration)
        if seed < 0:
            seed = int(torch.randint(0, 2**31 - 1, (1,)).item())
        generator = torch.Generator(device="cuda").manual_seed(seed)

        kwargs = dict(
            image=init_image,
            prompt=prompt,
            height=height,
            width=width,
            num_frames=num_frames,
            num_inference_steps=num_inference_steps,
            generator=generator,
            output=["videos", "audio", "sampling_rate"],
        )
        if last is not None:
            kwargs["last_image"] = last

        result = pipe(**kwargs)

        video_frames = result["videos"][0]
        audio_track = result["audio"][0]
        sampling_rate = result.get("sampling_rate", 32000)

        out_path = os.path.join(OUTPUT_DIR, f"{uuid.uuid4().hex}.mp4")
        encode_video(
            video_frames,
            fps=24,
            output_path=out_path,
            audio=audio_track,
            audio_sample_rate=sampling_rate,
        )

        with open(out_path, "rb") as f:
            data = f.read()
        return Response(content=data, media_type="video/mp4")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [ ]:
import subprocess
import time
import re

# Start FastAPI in the background
uvicorn_proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait briefly for server startup
time.sleep(5)

# Start Cloudflare Quick Tunnel and print the public URL
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n" + "="*60)
        print("YOUR CLOUDFLARE API URL:", match.group(0))
        print("="*60 + "\n")
        break